# Laboratorio 14. Transferencia de aprendizaje

**Pregunta guía:** ¿Qué se transfiere cuando congelamos un extractor visual aprendido?

## Objetivos
Al finalizar podrás entrenar una CNN en una tarea fuente; reutilizar y congelar el extractor; entrenar una cabeza nueva con pocos ejemplos objetivo.

## Prerrequisitos y conexión
Parte de la CNN del laboratorio 13; ahora reutilizamos representaciones entre tareas. Se requiere Python básico; cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Transfer learning inicializa una tarea objetivo con parámetros $\theta_s$ aprendidos en una tarea fuente. Al congelar el backbone $f_{\theta_s}$, solo se optimiza la cabeza $g_\phi$: $\min_\phi L(g_\phi(f_{\theta_s}(x)),y)$. Reducir parámetros entrenables no garantiza que la representación encaje con el dominio objetivo.

## Problema y datos
Dos tareas sintéticas de líneas comparten bordes pero tienen etiquetas distintas. Todo se genera localmente; no se descargan pesos. Todos los datos se generan en el notebook: no se requieren descargas ni archivos locales.

## Experimento conceptual
¿Cuánto aprende una cabeza nueva cuando el extractor ya representa bordes? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


PyTorch: 2.8.0+cpu | dispositivo: cpu


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Entrenar la tarea fuente
La tarea fuente distingue orientación; los mismos ejemplos contienen estructura visual reutilizable.


In [2]:
def make_line_images(count, task):
    samples, targets = [], []
    for index in range(count):
        label = index % 2; canvas = np.zeros((12, 12), dtype=np.float32)
        if task == 'orientation':
            if label == 0: canvas[5:7, 1:11] = 1
            else: canvas[1:11, 5:7] = 1
        else:
            if label == 0: canvas[2:4, 2:10] = 1
            else: canvas[8:10, 2:10] = 1
        canvas += np.random.normal(0, .15, canvas.shape).astype(np.float32)
        samples.append(np.clip(canvas, 0, 1)); targets.append(label)
    return torch.tensor(np.array(samples)[:, None], device=device), torch.tensor(targets, device=device)
source_x, source_y = make_line_images(300, 'orientation')
class TransferCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features = nn.Sequential(nn.Conv2d(1,8,3,padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2)); self.head = nn.Linear(144,2)
    def forward(self, inputs): return self.head(self.features(inputs).flatten(1))
source_model = TransferCNN().to(device); optimizer = torch.optim.Adam(source_model.parameters(), lr=.01)
for epoch in range(12):
    loss = F.cross_entropy(source_model(source_x), source_y); optimizer.zero_grad(); loss.backward(); optimizer.step()
print('fuente accuracy:', (source_model(source_x).argmax(1) == source_y).float().mean().item())


fuente accuracy: 1.0


**Resultado e interpretación.** El resultado fuente solo confirma que se entrenó un extractor en la tarea de orientación; aún no mide transferencia.


### Transferir con pocos datos
Congelamos el extractor y ajustamos una cabeza nueva; el split objetivo se realiza antes del ajuste.


In [3]:
target_x, target_y = make_line_images(100, 'position')
transfer_model = TransferCNN().to(device)
transfer_model.features.load_state_dict(source_model.features.state_dict())
for parameter in transfer_model.features.parameters(): parameter.requires_grad = False
few_train = torch.arange(20, device=device); target_test = torch.arange(20, len(target_y), device=device)
optimizer = torch.optim.Adam(transfer_model.head.parameters(), lr=.03)
for epoch in range(35):
    logits = transfer_model(target_x[few_train]); loss = F.cross_entropy(logits, target_y[few_train])
    optimizer.zero_grad(); loss.backward(); optimizer.step()
transfer_model.eval()
with torch.no_grad(): target_predictions = transfer_model(target_x[target_test]).argmax(1)
print('muestras de ajuste:', len(few_train), '| test transfer accuracy:', (target_predictions == target_y[target_test]).float().mean().item())
print('parámetros entrenables:', sum(p.numel() for p in transfer_model.parameters() if p.requires_grad))


muestras de ajuste: 20 | test transfer accuracy: 1.0
parámetros entrenables: 290


**Resultado e interpretación.** La prueba objetivo contiene ejemplos no usados para ajustar la cabeza. Son tareas sintéticas relacionadas; la conclusión no se extrapola directamente a fotografías reales.


## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, tamaño de muestra, arquitectura y protocolo. No extrapoles métricas sintéticas a datos reales; separa decisiones de modelado de conclusiones generales. La celda de salida aporta la evidencia numérica y visual para responder la pregunta guía.

- Los parámetros y activaciones determinan cómo se representa la señal.
- La pérdida, el optimizador y el protocolo de evaluación forman parte del experimento.
- Una métrica aislada no describe todos los errores ni garantiza generalización.

## Ejercicios progresivos
1. **Guiado:** Guiado: imprime `requires_grad` de cada capa antes y después de congelar.
2. **Independiente:** Independiente: compara con un extractor aleatorio congelado usando el mismo split y pasos.
3. **Desafío:** Desafío: propone una estrategia de fine-tuning parcial y define una comparación controlada.

## Conexión con el siguiente laboratorio
Usa la representación, operación o criterio de evaluación de este laboratorio como punto de partida del siguiente paso de la secuencia.
